# Sleep-EDF: physiology + representations124–146

Тот же launcher05. Input: исходный full_psg_v1 + новый physiology-inputs.zip.
Полный порядок: contrastive → yasa → features → events → blends → signals → calib → fusion_prepare → fusion → blends → final.
`all` выполняет этот порядок: сначала структурный smoke всей серии, затем full. Для обучения явно установите ALLOW_TRAINING=True.
YASA и LightGBM используют CPU; contrastive/dual-stream/fusion — две независимые очереди T4. Все результаты включают val и unlabelled test.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "c4f8813fc94cca235443009525a2e014556ba338"  # Версия кода закреплена для безопасного resume.
PROJECT_ROOT = Path("/tmp/sleep-edf-physiology-code-journal")
OUTPUT_ROOT = Path("/kaggle/working/physiology-results")
TRAINED_ROOT = Path("/kaggle/working/physiology-trained")
CACHE_ROOT = Path("/tmp/sleep-edf-physiology-cache")
INPUT_ROOTS = []  # Автопоиск нового sleep-edf-physiology-inputs.zip.
RESUME_ROOT = None  # Каталог или ZIP результатов этой серии.
RESUME_TRAINED_ROOT = None  # Каталог или ZIP весов этой серии.
RESUME_CACHE_ROOT = None  # Каталог или ZIP кэша событий/OOF этой серии.

RUN_GROUP = "features"  # features / yasa / contrastive / events / signals / calib / fusion_prepare / fusion / blends / final / all / custom
CUSTOM_IDS = []
SMOKE_ONLY = True  # False: сначала структурный smoke, затем полный train-CV -> refit -> val/test.
SMOKE_EPOCHS = 160
ALLOW_TRAINING = False  # True для contrastive/events/calib/fusion_prepare/fusion/all.
DEVICE = "cuda:0"
THREADS = max(1, os.cpu_count() or 2)
FEATURE_WORKERS = min(2, THREADS)  # Один общий индикатор; native YASA работает на CPU.
PARALLEL_DEVICES = [
    "cuda:0",
    "cuda:1",
]  # Две независимые очереди нейросетей; [] = последовательно.
EXPORT_CACHE = True
EXPORT_TRAINED = True

GROUPS = {
    "yasa": [124, 125],
    "contrastive": [126, 127, 128],
    "calib": [129, 130, 131],
    "events": [132, 133],
    "fusion": [134, 135],
    "blends": list(range(136, 146)),
    "final": [146],
}


def experiment_name(n):
    return f"{n:03d}_tree" if n in (132, 133) else f"{n:03d}_pretrained"


if RUN_GROUP not in [*GROUPS, "features", "signals", "fusion_prepare", "all", "custom"]:
    raise ValueError("Неизвестная RUN_GROUP")
if RUN_GROUP == "custom" and (
    not CUSTOM_IDS
    or any(n not in [experiment_name(i) for i in range(124, 147)] for n in CUSTOM_IDS)
):
    raise ValueError("CUSTOM_IDS: полные имена, например126_pretrained или132_tree")


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-refinement-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.physiology_run",
]
ENV.update(
    {
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
print(
    "Группа:", RUN_GROUP, "· ALLOW_TRAINING:", ALLOW_TRAINING, "· CPU threads:", THREADS
)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(path):
    path = Path(path)
    if path.is_dir():
        return path
    key = hashlib.sha256(
        (str(path) + str(path.stat().st_size) + str(path.stat().st_mtime_ns)).encode()
    ).hexdigest()[:16]
    target = Path("/tmp/sleep-edf-physiology-inputs") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(path) as z:
        for info in z.infolist():
            dest = (target / info.filename).resolve()
            if (
                not dest.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный путь в ZIP")
            z.extract(info, target)
    (target / ".unpacked").touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file() and (p.parent / "epochs/train").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1, найдено{roots}")
DATA_ROOT = roots[0]
if not INPUT_ROOTS:
    for p in Path("/kaggle/input").rglob("bundle.json"):
        if json.loads(p.read_text()).get("kind") == "physiology_inputs_v1":
            INPUT_ROOTS.append(p.parent)
    if not INPUT_ROOTS:
        for archive in Path("/kaggle/input").rglob("*.zip"):
            with ZipFile(archive) as z:
                if (
                    "bundle.json" in z.namelist()
                    and json.loads(z.read("bundle.json")).get("kind")
                    == "physiology_inputs_v1"
                ):
                    INPUT_ROOTS.append(unpack(archive))
INPUT_ROOTS = [unpack(p) for p in INPUT_ROOTS]
if len(INPUT_ROOTS) != 1:
    raise RuntimeError(
        "Подключите ровно один новый sleep-edf-physiology-inputs.zip / распакованный bundle"
    )


def restore(previous, target):
    if previous is None:
        return
    previous = unpack(previous)
    for p in previous.rglob("*"):
        if not p.is_file() or p.name == ".unpacked":
            continue
        dest = target / p.relative_to(previous)
        if dest.exists():
            if (
                hashlib.sha256(dest.read_bytes()).digest()
                != hashlib.sha256(p.read_bytes()).digest()
            ):
                raise ValueError(f"Resume conflict, существующий файл сохранён: {dest}")
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dest)


for previous, target in [
    (RESUME_ROOT, OUTPUT_ROOT),
    (RESUME_TRAINED_ROOT, TRAINED_ROOT),
    (RESUME_CACHE_ROOT, CACHE_ROOT),
]:
    restore(previous, target)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--cache-root",
    str(CACHE_ROOT),
    "--trained-root",
    str(TRAINED_ROOT),
    "--input-root",
    str(INPUT_ROOTS[0]),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
    "--workers",
    str(FEATURE_WORKERS),
]
if ALLOW_TRAINING:
    COMMON += ["--allow-training"]
print("DATA_ROOT:", DATA_ROOT, "\nINPUT_ROOT:", INPUT_ROOTS[0])
print(
    "Прежние134 embeddings и106 признаки уже внутри bundle. Нового backbone inference для них нет."
)
print(
    "fusion_prepare:6 готовых nested экспертов +6 новых небольших upstream heads. Это отдельный более дорогой этап."
)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

from src.notebook import cancel_commands, reset_commands

# Recover only index/CSV publication from complete saved results. No model runs.
reset_commands()
run_command(
    [*BASE[:-1], "src.result_recovery", "--output-root", str(OUTPUT_ROOT)],
    cwd=PROJECT_ROOT,
    env=ENV,
)

PREPARE = {"features": "features", "signals": "signals", "fusion_prepare": "fusion"}


def run_one(names, smoke, prepare=None, device=None, tag="main"):
    command = [
        *BASE,
        *[str(PROJECT_ROOT / "configs" / (n + ".yaml")) for n in names],
        *COMMON,
        "--report-tag",
        tag,
    ]
    env = ENV.copy()
    if device is not None:
        command[command.index("--device") + 1] = device
        threads = max(1, THREADS // 2)
        command[command.index("--threads") + 1] = str(threads)
        command[command.index("--workers") + 1] = "1"
        env.update(
            {
                k: str(threads)
                for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
            }
        )
    if smoke:
        command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
    if prepare:
        command += ["--prepare", prepare]
    started = time.time_ns()
    try:
        run_command(command, cwd=PROJECT_ROOT, env=env)
    except subprocess.CalledProcessError:
        if prepare:
            raise
        print(
            "Ошибки сохранены в<ID>/last_error.json; завершённые результаты не потеряны."
        )
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    report = namespace / f"physiology_failures-{tag}.json"
    if not report.exists() or report.stat().st_mtime_ns < started:
        return []
    failed = {r["experiment_id"] for r in json.loads(report.read_text())}
    return [
        n
        for n in names
        if n not in failed and (namespace / n / "metrics.json").exists()
    ]


def run_group(group, smoke):
    reset_commands()
    if group in PREPARE:
        run_one([], smoke, PREPARE[group])
        return
    names = (
        CUSTOM_IDS if group == "custom" else [experiment_name(n) for n in GROUPS[group]]
    )
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    if not smoke:
        names = [
            n for n in names if (OUTPUT_ROOT / "smoke" / n / "metrics.json").exists()
        ]
    if group == "blends":
        # Already completed independent directions can be blended immediately;
        # missing directions are left for the later blends pass, not marked failed.
        names = [
            n
            for n in names
            if (
                namespace / experiment_name(124 + int(n[:3]) - 136) / "metrics.json"
            ).exists()
        ]
    if group in ("contrastive", "calib", "fusion") and len(PARALLEL_DEVICES) == 2:
        if len(set(PARALLEL_DEVICES)) != 2:
            raise ValueError("Устройства должны различаться")
        if not (OUTPUT_ROOT / "implementation.json").exists():
            run_one([], smoke)
        pool = ThreadPoolExecutor(max_workers=2)
        try:
            futures = [
                pool.submit(
                    run_one, names[i::2], smoke, None, device, f"{group}-gpu{i}"
                )
                for i, device in enumerate(PARALLEL_DEVICES)
                if names[i::2]
            ]
            for future in futures:
                future.result()
        except BaseException:
            cancel_commands()
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)
    elif names:
        run_one(names, smoke, tag=group)


if (
    RUN_GROUP in ("contrastive", "events", "calib", "fusion_prepare", "fusion", "all")
    and not ALLOW_TRAINING
):
    raise ValueError("Для выбранной группы нужно ALLOW_TRAINING=True")
order = (
    [
        "contrastive",
        "yasa",
        "features",
        "events",
        "blends",
        "signals",
        "calib",
        "fusion_prepare",
        "fusion",
        "blends",
        "final",
    ]
    if RUN_GROUP == "all"
    else [RUN_GROUP]
)
for smoke in [True] if SMOKE_ONLY else [True, False]:
    for group in order:
        print(("SMOKE" if smoke else "FULL"), "· группа", group)
        run_group(group, smoke)
run_command(
    [*BASE[:-1], "src.result_recovery", "--output-root", str(OUTPUT_ROOT)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
rows = []
for n in range(124, 147):
    name = experiment_name(n)
    path = namespace / name / "metrics.json"
    if path.exists():
        m = json.loads(path.read_text())
        rows.append(
            {
                "ID": name,
                "mode": "SMOKE" if m["smoke"] else "VAL+TEST",
                "val_macro_f1": m["validation"]["macro_f1"],
                "fit_s": m["training_seconds"],
                "inference_s": m["inference_seconds"],
            }
        )
    elif (namespace / name / "last_error.json").exists():
        rows.append(
            {
                "ID": name,
                "mode": "FAILED/PARTIAL",
                "error": json.loads((namespace / name / "last_error.json").read_text())[
                    "error"
                ],
            }
        )
display(pd.DataFrame(rows))

Последняя ячейка доступна после Stop. Все три ZIP накопительные, без исходных сигналов и временных signal pools.
Для новой сессии подключите скачанные ZIP и задайте RESUME_ROOT / RESUME_TRAINED_ROOT / RESUME_CACHE_ROOT: можно указать ZIP либо распакованный каталог. Версия кода должна совпадать.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

from src.notebook import reset_commands

reset_commands()
run_command(
    [*BASE[:-1], "src.result_recovery", "--output-root", str(OUTPUT_ROOT)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
for enabled, option, label in [
    (True, "--export", "results"),
    (EXPORT_CACHE, "--export-cache", "cache"),
    (EXPORT_TRAINED, "--export-trained", "trained"),
]:
    if not enabled:
        continue
    if (
        label == "trained"
        and not any(TRAINED_ROOT.rglob("*.pt"))
        and not any(TRAINED_ROOT.rglob("*.joblib"))
    ):
        continue
    target = Path("/kaggle/working") / f"sleep-edf-physiology-{label}-{stamp}.zip"
    run_command(
        [
            *BASE,
            "--output-root",
            str(OUTPUT_ROOT),
            "--cache-root",
            str(CACHE_ROOT),
            "--trained-root",
            str(TRAINED_ROOT),
            option,
            str(target),
        ],
        cwd=PROJECT_ROOT,
        env=ENV,
    )
    display(
        FileLink(
            str(target.relative_to(Path.cwd())),
            result_html_prefix=f"Накопительный ZIP {label}: ",
        )
    )
print(
    "results: метрики/probabilities/submissions/код; cache: события/OOF без signal pools; trained: best/last и LightGBM."
)